# Technical Data Labeler
This notebook pulls historical OHLCV data and engineers features based on price action, such as Moving Averages, Volatility, and RSI. 

First, we will import our libraries and define the functions needed to fetch the data and calculate these technical indicators. We also define our target variable: the 5-day future percentage return.

In [1]:
import pandas as pd
import numpy as np
import os

# Suppress SettingWithCopyWarning for cleaner output
pd.options.mode.chained_assignment = None  

def load_local_data(filepath, company_symbol):
    """Loads local CSV data and formats it for technical analysis."""
    df = pd.read_csv(filepath)
    
    # Standardize column names to Title Case
    df.rename(columns={'date': 'Date', 'open': 'Open', 'high': 'High', 'low': 'Low', 'close': 'Close', 'volume': 'Volume'}, inplace=True)
    
    # Set Date as index if not already
    if 'Date' in df.columns:
        df['Date'] = pd.to_datetime(df['Date'])
        df.set_index('Date', inplace=True)
    df.sort_index(inplace=True)
    
    # Add company column
    df['Company'] = company_symbol
    return df

def add_technical_features(df):
    """Calculates Moving Averages, RSI, and Volatility."""
    df['SMA_10'] = df['Close'].rolling(window=10).mean()
    df['SMA_50'] = df['Close'].rolling(window=50).mean()
    
    df['Daily_Return'] = df['Close'].pct_change()
    df['Volatility'] = df['Daily_Return'].rolling(window=20).std()
    
    # RSI Calculation
    delta = df['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / loss
    df['RSI'] = 100 - (100 / (1 + rs))
    
    df.dropna(inplace=True)
    return df

def create_target(df):
    """
    Defines the target variable: the percentage change over the next 5 days.
    Splits data into 'labeled' (for training) and 'live' (for real-time predictions).
    """
    df['Target_5d_Return'] = df['Close'].pct_change(periods=5).shift(-5)
    
    labeled_data = df.dropna()
    live_data = df[df['Target_5d_Return'].isna()]
    
    return labeled_data, live_data

Now, let's run the pipeline for all companies in our `csv-history` folder and combine them into single datasets.

In [2]:
history_dir = "../csv-history"
all_files = os.listdir(history_dir)

labeled_dfs = []
live_dfs = []

for file in all_files:
    if file.endswith(".csv") and not file.startswith("perfectly_balanced") and not file.startswith("technical_data"):
        symbol = file.replace(".csv", "").upper()
        print(f"Processing {symbol}...")
        
        filepath = os.path.join(history_dir, file)
        data = load_local_data(filepath, symbol)
        data = add_technical_features(data)
        labeled, live = create_target(data)
        
        labeled_dfs.append(labeled)
        live_dfs.append(live)

# Concatenate all dataframes sequentially (keeps data sorted chronologically per company)
labeled_data = pd.concat(labeled_dfs)
live_data = pd.concat(live_dfs)

print(f"\nTotal labeled rows: {len(labeled_data)}")
labeled_data.head()

Processing AAPL...
Processing AMZN...
Processing BRKL...
Processing GOOGL...
Processing JPM...
Processing LLY...
Processing MSFT...
Processing NVDA...
Processing TSLA...
Processing WMT...

Total labeled rows: 82742


,Open,High,Low,Close,adj close,Volume,Company,SMA_10,SMA_50,Daily_Return,Volatility,RSI,Target_5d_Return
Date,,,,,,,,,,,,,
1981-02-24,0.428571,0.428571,0.424107,0.424107,0.336037,4244800,AAPL,0.458705,0.530312,-0.035533,0.033942,31.325304,0.105263
1981-02-25,0.450893,0.453125,0.450893,0.450893,0.357260,4872000,AAPL,0.455134,0.529062,0.063158,0.038142,34.482759,0.029703
1981-02-26,0.457589,0.459821,0.457589,0.457589,0.362566,2710400,AAPL,0.453795,0.528482,0.014851,0.038252,36.666668,0.009756
1981-02-27,0.473214,0.477679,0.473214,0.473214,0.374946,3690400,AAPL,0.454464,0.528929,0.034146,0.038824,40.625003,-0.033019
1981-03-02,0.475446,0.477679,0.475446,0.475446,0.376715,2940000,AAPL,0.456473,0.529196,0.004717,0.037098,47.058827,-0.112676


Finally, we save this labeled data and the live data to our shared `csv-history` folder so the Technical Agent can access it for training.

In [3]:
labeled_data.to_csv("../csv-history/technical_data_prepared.csv")
live_data.to_csv("../csv-history/technical_data_live.csv") 
print("Data prepped and saved to the csv-history folder!")

Data prepped and saved to the csv-history folder!
